# ⚡ AN-RA — SOVEREIGN AI MASTER NOTEBOOK

```
  ╔═══════════════════════════════════════════════════╗
  ║   AN-RA · Transformer Language Model Control      ║
  ║   Single notebook. Full lifecycle. Zero friction. ║
  ╚═══════════════════════════════════════════════════╝
```

**Run order — Fresh session:**  
`Cell 1` → `Cell 2` → `Cell 3` → `Cell 4` → `Cell 5` → `Cell 6`

**Run order — Resumed session (checkpoint exists on Drive):**  
`Cell 1` → `Cell 2` → `Cell 5` → `Cell 6`

**Save before closing:**  
`Cell 7` — always run this before the session times out.

---

## 🔧 Cell 1 — SETUP
*Mounts Drive, clones repo, installs dependencies, prints system info.*  
*Run every session. Safe to re-run.*

In [ ]:
# ==============================================================
# CELL 1 — SETUP
# Mounts Google Drive, clones your GitHub repo, installs deps,
# and prints full system info. Safe to re-run each session.
# ==============================================================

import subprocess, sys, os, platform, builtins as _b

# ── 1a. Mount Google Drive ─────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_DIR = '/content/drive/MyDrive/AnRa'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'[Drive] ✓ Mounted. AnRa directory: {DRIVE_DIR}')

# ── 1b. Clone / pull GitHub repo ──────────────────────────────
# Option A: set REPO_URL directly below (recommended for repeated use)
# Option B: leave as empty string — will prompt at runtime
REPO_URL = ''                    # ← paste your repo URL here to skip the prompt
REPO_DIR = '/content/anra'

# Check Drive for a persisted repo URL from a previous session
_repo_url_cache = os.path.join(DRIVE_DIR, '.repo_url')
if not REPO_URL and os.path.exists(_repo_url_cache):
    with open(_repo_url_cache) as f:
        REPO_URL = f.read().strip()
    print(f'[Repo] Loaded cached URL: {REPO_URL}')

if not REPO_URL:
    REPO_URL = input('Enter your GitHub repo URL (e.g. https://github.com/user/anra): ').strip()
    # Persist for future sessions
    with open(_repo_url_cache, 'w') as f:
        f.write(REPO_URL)
    print(f'[Repo] URL saved to Drive for future sessions.')

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('[Repo] Already cloned — pulling latest changes...')
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print(result.stdout.strip() or '[Repo] Already up to date.')
else:
    print(f'[Repo] Cloning {REPO_URL} ...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print(f'[Repo] ✓ Clone complete.')

os.chdir(REPO_DIR)
_b.ANRA_REPO_DIR = REPO_DIR      # make available to all downstream cells
_b.ANRA_DRIVE_DIR = DRIVE_DIR
print(f'[Repo] Working directory: {os.getcwd()}')

# ── 1c. Install dependencies ───────────────────────────────────
print('\n[Deps] Installing dependencies (output suppressed)...')
deps = [
    'torch',          # transformer model
    'pyngrok==7.2.0', # pinned for stability — ngrok tunnel
    'flask',          # API server
    'flask-cors',     # cross-origin requests
    'requests',       # health checks & chat client
    'numpy',          # numerical ops
    'tqdm',           # progress bars
]

# Check if a requirements.txt exists in repo and merge
_req_path = os.path.join(REPO_DIR, 'requirements.txt')
if os.path.exists(_req_path):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', _req_path], check=True)
    print('[Deps] ✓ Installed from requirements.txt')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + deps, check=True)
print('[Deps] ✓ All core dependencies installed.')

# ── 1d. System info ────────────────────────────────────────────
import torch

print('\n' + '='*52)
print('  SYSTEM INFORMATION')
print('='*52)
print(f'  Python  : {platform.python_version()}')
print(f'  PyTorch : {torch.__version__}')
print(f'  CUDA    : {torch.version.cuda or "N/A"}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU     : {gpu_name} ({gpu_mem:.1f} GB VRAM)')
    _b.ANRA_DEVICE = 'cuda'
else:
    print('  GPU     : None — CPU only (training will be slow)')
    _b.ANRA_DEVICE = 'cpu'

try:
    with open('/proc/meminfo') as _f:
        for _line in _f:
            if 'MemTotal' in _line:
                _ram_gb = int(_line.split()[1]) / 1e6
                print(f'  RAM     : {_ram_gb:.1f} GB')
                break
except Exception:
    pass

print('='*52)
print('\n[Setup] ✓ Cell 1 complete. Proceed to Cell 2.')


---
## 🔄 Cell 2 — SESSION RESTORE
*Checks Drive for existing checkpoints and tokenizer.*  
*Prefers identity checkpoint over base checkpoint.*  
*Sets session flags used by all downstream cells.*

In [ ]:
# ==============================================================
# CELL 2 — SESSION RESTORE
# Scans Drive for checkpoints and tokenizer, copies to repo dir.
# Sets builtins flags: ANRA_CHECKPOINT, ANRA_HAS_TOKENIZER
# that downstream cells read to skip unnecessary work.
# ==============================================================

import os, shutil, builtins as _b

REPO_DIR  = getattr(_b, 'ANRA_REPO_DIR',  '/content/anra')
DRIVE_DIR = getattr(_b, 'ANRA_DRIVE_DIR', '/content/drive/MyDrive/AnRa')

# Checkpoints in priority order — best available wins
CHECKPOINT_PRIORITY = [
    'anra_brain_identity.pt',   # fine-tuned identity model (preferred)
    'anra_brain.pt',            # base trained model
]
TOKENIZER_FILE = 'tokenizer.pkl'

restored_checkpoint = None
restored_tokenizer  = False

print('[Restore] Scanning Google Drive for AnRa artifacts...')
print(f'[Restore] Drive path: {DRIVE_DIR}')

# ── Checkpoint restore ─────────────────────────────────────────
for fname in CHECKPOINT_PRIORITY:
    drive_path = os.path.join(DRIVE_DIR, fname)
    local_path = os.path.join(REPO_DIR, fname)
    if os.path.exists(drive_path):
        size_mb = os.path.getsize(drive_path) / 1e6
        print(f'[Restore] Found: {fname} ({size_mb:.1f} MB) — copying...')
        shutil.copy2(drive_path, local_path)
        print(f'[Restore] ✓ Restored checkpoint: {fname}')
        restored_checkpoint = fname
        break   # stop at highest-priority match
    else:
        print(f'[Restore]   Not found: {fname}')

if not restored_checkpoint:
    print('[Restore] ✗ No checkpoint on Drive — will train from scratch (run Cell 3).')

# ── Tokenizer restore ──────────────────────────────────────────
drive_tok = os.path.join(DRIVE_DIR, TOKENIZER_FILE)
local_tok  = os.path.join(REPO_DIR, TOKENIZER_FILE)
if os.path.exists(drive_tok):
    shutil.copy2(drive_tok, local_tok)
    tok_size = os.path.getsize(local_tok) / 1e3
    print(f'[Restore] ✓ Tokenizer restored: {TOKENIZER_FILE} ({tok_size:.1f} KB)')
    restored_tokenizer = True
else:
    print('[Restore] ✗ No tokenizer on Drive — will create during training.')

# ── Also list any timestamped backups ─────────────────────────
backups = [
    f for f in os.listdir(DRIVE_DIR)
    if f.endswith('.pt') and '_2' in f   # matches e.g. anra_brain_20240101_120000.pt
]
if backups:
    backups.sort(reverse=True)
    print(f'\n[Restore] Available backups on Drive ({len(backups)} found):')
    for bk in backups[:5]:   # show latest 5
        bk_mb = os.path.getsize(os.path.join(DRIVE_DIR, bk)) / 1e6
        print(f'          {bk} ({bk_mb:.1f} MB)')
    if len(backups) > 5:
        print(f'          ...and {len(backups) - 5} more.')

# ── Set session flags ─────────────────────────────────────────
_b.ANRA_CHECKPOINT    = restored_checkpoint
_b.ANRA_HAS_TOKENIZER = restored_tokenizer

print()
print('='*48)
print('  SESSION RESTORE SUMMARY')
print('='*48)
print(f'  Active checkpoint : {restored_checkpoint or "None (train from scratch)"}')
print(f'  Tokenizer ready   : {restored_tokenizer}')
print(f'  Device            : {getattr(_b, "ANRA_DEVICE", "unknown")}')
print('='*48)

if restored_checkpoint:
    print('\n[Restore] ✓ Ready to launch. You can skip to Cell 5.')
else:
    print('\n[Restore] → Run Cell 3 to train from scratch.')


---
## 🧠 Cell 3 — BASE TRAINING *(optional)*
*Only runs if no checkpoint exists, or if `FORCE_RETRAIN = True`.*  
*Executes `build_anra_brain.py` and auto-saves result to Drive.*

In [ ]:
# ==============================================================
# CELL 3 — BASE TRAINING  (optional)
# Skipped automatically if a checkpoint was found in Cell 2.
# Set FORCE_RETRAIN = True to train even with an existing model.
# Auto-saves checkpoint + tokenizer to Drive when complete.
# ==============================================================

import os, subprocess, shutil, sys, builtins as _b

# ── Configuration ──────────────────────────────────────────────
FORCE_RETRAIN = False    # ← set True to retrain even if checkpoint exists

REPO_DIR     = getattr(_b, 'ANRA_REPO_DIR',  '/content/anra')
DRIVE_DIR    = getattr(_b, 'ANRA_DRIVE_DIR', '/content/drive/MyDrive/AnRa')
TRAIN_SCRIPT = os.path.join(REPO_DIR, 'build_anra_brain.py')
OUTPUT_CKPT  = os.path.join(REPO_DIR, 'anra_brain.pt')
OUTPUT_TOK   = os.path.join(REPO_DIR, 'tokenizer.pkl')

# ── Guard: skip if already have checkpoint ────────────────────
has_ckpt = getattr(_b, 'ANRA_CHECKPOINT', None)

if has_ckpt and not FORCE_RETRAIN:
    print(f'[Train] Checkpoint "{has_ckpt}" already loaded.')
    print('[Train] Skipping base training.')
    print('[Train] → Set FORCE_RETRAIN = True at the top of this cell to override.')
else:
    # ── Validate script exists ────────────────────────────────
    if not os.path.exists(TRAIN_SCRIPT):
        print(f'[Train] ✗ ERROR: {TRAIN_SCRIPT} not found.')
        print('[Train]   Ensure build_anra_brain.py is committed to your repo.')
        raise FileNotFoundError(TRAIN_SCRIPT)

    print('[Train] Starting base training...')
    print(f'[Train] Script : {TRAIN_SCRIPT}')
    print(f'[Train] Device : {getattr(_b, "ANRA_DEVICE", "cpu")}')
    print('[Train] ' + '-'*45)

    import time
    _t0 = time.time()

    # Stream output live to the notebook cell
    result = subprocess.run(
        [sys.executable, TRAIN_SCRIPT],
        cwd=REPO_DIR
        # No capture_output — streams directly to notebook
    )

    _elapsed = time.time() - _t0
    print('[Train] ' + '-'*45)
    print(f'[Train] Elapsed: {_elapsed/60:.1f} minutes')

    if result.returncode != 0:
        print(f'[Train] ✗ Script exited with code {result.returncode}')
        raise RuntimeError(f'Training failed (exit code {result.returncode})')

    # ── Auto-save checkpoint to Drive ─────────────────────────
    if os.path.exists(OUTPUT_CKPT):
        drive_dest = os.path.join(DRIVE_DIR, 'anra_brain.pt')
        shutil.copy2(OUTPUT_CKPT, drive_dest)
        size_mb = os.path.getsize(drive_dest) / 1e6
        print(f'[Train] ✓ Checkpoint saved to Drive: anra_brain.pt ({size_mb:.1f} MB)')
        _b.ANRA_CHECKPOINT = 'anra_brain.pt'
    else:
        print(f'[Train] ✗ anra_brain.pt not found after training. Check your script.')

    # ── Auto-save tokenizer to Drive ──────────────────────────
    if os.path.exists(OUTPUT_TOK):
        shutil.copy2(OUTPUT_TOK, os.path.join(DRIVE_DIR, 'tokenizer.pkl'))
        print('[Train] ✓ Tokenizer saved to Drive: tokenizer.pkl')
        _b.ANRA_HAS_TOKENIZER = True
    else:
        print('[Train] ⚠ tokenizer.pkl not produced — your script may handle it differently.')

    print(f'\n[Train] ✓ Cell 3 complete. Proceed to Cell 4 (fine-tune) or Cell 5 (launch).')


---
## 🪐 Cell 4 — IDENTITY FINE-TUNING *(optional)*
*Runs `finetune_anra.py` to imprint An-Ra's sovereign character.*  
*Requires `identity_data.txt` in repo root (or set `IDENTITY_DATA` path below).*  
*Auto-saves `anra_brain_identity.pt` to Drive.*

In [ ]:
# ==============================================================
# CELL 4 — IDENTITY FINE-TUNING  (optional)
# Loads the base checkpoint, fine-tunes on identity dialogue,
# and saves anra_brain_identity.pt to Drive.
# Set IDENTITY_DATA to the path of your fine-tuning text file.
# ==============================================================

import os, subprocess, shutil, sys, builtins as _b

# ── Configuration ──────────────────────────────────────────────
IDENTITY_DATA   = os.path.join(
    getattr(_b, 'ANRA_REPO_DIR', '/content/anra'),
    'identity_data.txt'
)  # ← change path if your file is elsewhere

REPO_DIR        = getattr(_b, 'ANRA_REPO_DIR',  '/content/anra')
DRIVE_DIR       = getattr(_b, 'ANRA_DRIVE_DIR', '/content/drive/MyDrive/AnRa')
FINETUNE_SCRIPT = os.path.join(REPO_DIR, 'finetune_anra.py')
OUTPUT_CKPT     = os.path.join(REPO_DIR, 'anra_brain_identity.pt')

# ── Validate prerequisites ────────────────────────────────────
_ok = True

if not os.path.exists(FINETUNE_SCRIPT):
    print(f'[Finetune] ✗ Script not found: {FINETUNE_SCRIPT}')
    _ok = False

# Check Drive for identity data as fallback
if not os.path.exists(IDENTITY_DATA):
    _drive_id = os.path.join(DRIVE_DIR, 'identity_data.txt')
    if os.path.exists(_drive_id):
        shutil.copy2(_drive_id, IDENTITY_DATA)
        print(f'[Finetune] Loaded identity_data.txt from Drive.')
    else:
        print(f'[Finetune] ✗ Identity data not found at: {IDENTITY_DATA}')
        print('[Finetune]   Options:')
        print(f'[Finetune]   1. Commit identity_data.txt to your repo')
        print(f'[Finetune]   2. Upload to Drive at: {_drive_id}')
        print(f'[Finetune]   3. Set IDENTITY_DATA path at top of this cell')
        _ok = False

base_ckpt = getattr(_b, 'ANRA_CHECKPOINT', None)
if not base_ckpt:
    print('[Finetune] ✗ No base checkpoint. Run Cell 2 or Cell 3 first.')
    _ok = False

if _ok:
    _base_path = os.path.join(REPO_DIR, base_ckpt)
    print(f'[Finetune] Base checkpoint : {base_ckpt}')
    print(f'[Finetune] Identity data   : {IDENTITY_DATA}')
    print(f'[Finetune] Output          : anra_brain_identity.pt')
    print('[Finetune] ' + '-'*43)

    import time
    _t0 = time.time()

    # finetune_anra.py expected signature:
    #   python finetune_anra.py --checkpoint <path> --data <path> --output <path>
    # Adjust args below if your script has a different CLI interface.
    result = subprocess.run(
        [
            sys.executable, FINETUNE_SCRIPT,
            '--checkpoint', _base_path,
            '--data',       IDENTITY_DATA,
            '--output',     OUTPUT_CKPT,
        ],
        cwd=REPO_DIR
        # No capture_output — streams live to notebook
    )

    print('[Finetune] ' + '-'*43)
    print(f'[Finetune] Elapsed: {(time.time()-_t0)/60:.1f} minutes')

    if result.returncode == 0 and os.path.exists(OUTPUT_CKPT):
        drive_dest = os.path.join(DRIVE_DIR, 'anra_brain_identity.pt')
        shutil.copy2(OUTPUT_CKPT, drive_dest)
        size_mb = os.path.getsize(drive_dest) / 1e6
        print(f'\n[Finetune] ✓ Fine-tuning complete.')
        print(f'[Finetune] ✓ Saved to Drive: anra_brain_identity.pt ({size_mb:.1f} MB)')
        _b.ANRA_CHECKPOINT = 'anra_brain_identity.pt'
    elif result.returncode != 0:
        print(f'[Finetune] ✗ Script exited with code {result.returncode}')
    else:
        print(f'[Finetune] ✗ Script succeeded but {OUTPUT_CKPT} was not produced.')

print('\n[Finetune] ✓ Cell 4 complete. Proceed to Cell 5.')


---
## 🌐 Cell 5 — LAUNCH API
*Starts `app.py` in a background thread.*  
*Opens a public ngrok tunnel and prints the live URL.*  
*Runs a health check through the tunnel to confirm the API is alive.*

> **ngrok token:** Free at [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)

In [ ]:
# ==============================================================
# CELL 5 — LAUNCH API
# Starts app.py in a daemon thread, waits for it to be ready,
# opens an ngrok HTTPS tunnel, and confirms via health check.
# Stores the public URL in builtins.ANRA_PUBLIC_URL.
# ==============================================================

import os, sys, time, threading, requests, builtins as _b

# ── Configuration ──────────────────────────────────────────────
PORT = 5000

# ngrok auth token — paste yours here, or leave '' to be prompted.
# Get a free token at: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ''    # ← paste token here to avoid prompt each session

REPO_DIR  = getattr(_b, 'ANRA_REPO_DIR',  '/content/anra')
DRIVE_DIR = getattr(_b, 'ANRA_DRIVE_DIR', '/content/drive/MyDrive/AnRa')

# ── Load / prompt for ngrok token ─────────────────────────────
# Cache token to Drive so you only paste it once per account
_tok_cache = os.path.join(DRIVE_DIR, '.ngrok_token')
if not NGROK_TOKEN and os.path.exists(_tok_cache):
    with open(_tok_cache) as f:
        NGROK_TOKEN = f.read().strip()
    print('[API] Loaded ngrok token from Drive cache.')

if not NGROK_TOKEN:
    NGROK_TOKEN = input('Paste your ngrok auth token: ').strip()
    with open(_tok_cache, 'w') as f:
        f.write(NGROK_TOKEN)
    print('[API] Token saved to Drive for future sessions.')

# ── Kill any stale tunnels or servers from previous runs ───────
from pyngrok import ngrok, conf
ngrok.kill()   # closes any existing pyngrok tunnels
conf.get_default().auth_token = NGROK_TOKEN

# ── Validate app.py ────────────────────────────────────────────
APP_SCRIPT = os.path.join(REPO_DIR, 'app.py')
if not os.path.exists(APP_SCRIPT):
    raise FileNotFoundError(f'{APP_SCRIPT} not found. Ensure it is committed to your repo.')

# ── Set checkpoint env var for app.py to pick up ──────────────
active_ckpt = getattr(_b, 'ANRA_CHECKPOINT', None)
if active_ckpt:
    ckpt_full = os.path.join(REPO_DIR, active_ckpt)
    os.environ['ANRA_CHECKPOINT'] = ckpt_full
    os.environ['ANRA_TOKENIZER']  = os.path.join(REPO_DIR, 'tokenizer.pkl')
    print(f'[API] Checkpoint : {active_ckpt}')
else:
    print('[API] ⚠ No checkpoint set in ANRA_CHECKPOINT — app.py will use its default.')

# ── Start app.py in a background daemon thread ────────────────
os.chdir(REPO_DIR)
_server_log  = []
_server_errors = []

def _run_server():
    """Runs app.py as subprocess; captures stderr for error reporting."""
    import subprocess
    proc = subprocess.Popen(
        [sys.executable, APP_SCRIPT, '--port', str(PORT)],
        cwd=REPO_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,   # merge stderr into stdout
        text=True,
        bufsize=1
    )
    _b.ANRA_SERVER_PROC = proc      # store so Cell 7 can terminate cleanly
    for line in proc.stdout:
        _server_log.append(line.rstrip())
        if 'error' in line.lower() or 'traceback' in line.lower():
            _server_errors.append(line.rstrip())

_server_thread = threading.Thread(target=_run_server, daemon=True)
_server_thread.start()
print(f'[API] Server starting on localhost:{PORT}...')

# ── Poll until server responds or errors out ───────────────────
MAX_WAIT = 40   # seconds
_server_up = False
for i in range(MAX_WAIT):
    time.sleep(1)
    if _server_errors:
        print(f'\n[API] ✗ Server error detected:')
        for _e in _server_errors[:10]:
            print(f'       {_e}')
        break
    try:
        r = requests.get(f'http://localhost:{PORT}/health', timeout=2)
        if r.status_code == 200:
            print(f'[API] ✓ Server ready after {i+1}s.')
            _server_up = True
            break
    except requests.exceptions.ConnectionError:
        pass    # still starting up
    print(f'[API]   Waiting for server... {i+1}/{MAX_WAIT}s', end='\r')
else:
    print(f'\n[API] ✗ Server did not respond in {MAX_WAIT}s.')
    print('[API]   Last 5 server log lines:')
    for _line in _server_log[-5:]:
        print(f'       {_line}')

# ── Open ngrok tunnel ─────────────────────────────────────────
if _server_up:
    _tunnel = ngrok.connect(PORT, 'http')
    _public_url = _tunnel.public_url
    _b.ANRA_PUBLIC_URL = _public_url

    print()
    print('\033[1m' + '='*55 + '\033[0m')
    print('  ⚡  AN-RA IS LIVE')
    print('='*55)
    print(f'  Public URL  : {_public_url}')
    print(f'  Chat API    : {_public_url}/chat')
    print(f'  Health      : {_public_url}/health')
    print(f'  Local port  : http://localhost:{PORT}')
    print('='*55)

    # ── Public tunnel health check ─────────────────────────────
    time.sleep(3)   # give tunnel a moment to fully propagate
    try:
        _hc = requests.get(f'{_public_url}/health', timeout=15)
        print(f'\n[API] ✓ Public health check: HTTP {_hc.status_code}')
        print(f'[API]   Response: {_hc.text[:120]}')
    except Exception as _he:
        print(f'\n[API] ⚠ Public health check failed: {_he}')
        print('[API]   Tunnel may still be propagating — try again in 10s.')
else:
    print('[API] ✗ Skipping tunnel — server is not running.')
    print('[API]   Fix the server errors above, then re-run this cell.')

print('\n[API] ✓ Cell 5 complete. Proceed to Cell 6 to chat.')


---
## 💬 Cell 6 — LIVE CHAT *(inline)*
*Interactive Python loop — no external UI needed.*  
*Sends messages to the local `/chat` endpoint and prints responses.*  
*Type `quit` to exit, `url` to switch to the public tunnel.*

In [ ]:
# ==============================================================
# CELL 6 — LIVE CHAT  (inline, no external UI)
# Sends messages to An-Ra's /chat endpoint and prints replies.
# Uses the local endpoint by default (lowest latency).
# Commands:
#   quit       — exit the chat loop
#   url        — switch to the public ngrok URL
#   local      — switch back to local endpoint
#   clear      — clear conversation history
#   status     — print server and tunnel status
# ==============================================================

import requests, json, builtins as _b, time

PORT       = 5000
LOCAL_URL  = f'http://localhost:{PORT}/chat'
PUBLIC_URL = getattr(_b, 'ANRA_PUBLIC_URL', None)

# Start on local endpoint — always prefer it for speed
_current_url = LOCAL_URL
_history     = []   # stores {'role': 'user'/'assistant', 'content': '...'}

def _send(message: str, url: str, history: list) -> str:
    """POST a message to An-Ra and return the response string."""
    try:
        payload = {
            'message': message,
            'history': history,   # pass conversation history if your app supports it
        }
        resp = requests.post(url, json=payload, timeout=90)
        resp.raise_for_status()
        data = resp.json()
        # Support multiple common response shapes from app.py
        return (
            data.get('response') or
            data.get('text')     or
            data.get('reply')    or
            data.get('output')   or
            data.get('message')  or
            json.dumps(data, indent=2)
        )
    except requests.exceptions.ConnectionError:
        return '[ERROR] Cannot reach An-Ra. Is Cell 5 running?'
    except requests.exceptions.Timeout:
        return '[ERROR] Timeout — model may be generating a long response. Try a shorter prompt.'
    except requests.exceptions.HTTPError as e:
        return f'[ERROR] HTTP {e.response.status_code}: {e.response.text[:200]}'
    except Exception as e:
        return f'[ERROR] {type(e).__name__}: {e}'

# ── Print header ──────────────────────────────────────────────
print('='*52)
print('  ⚡  AN-RA LIVE CHAT')
print('='*52)
print(f'  Endpoint : {_current_url}')
print(f'  Public   : {PUBLIC_URL or "(not set — run Cell 5)"}')
print('  Commands : quit | url | local | clear | status')
print('='*52 + '\n')

# ── Chat loop ─────────────────────────────────────────────────
while True:
    try:
        _user_input = input('You: ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\n[Chat] Interrupted. An-Ra remains running.')
        break

    if not _user_input:
        continue

    # ── Handle commands ───────────────────────────────────────
    if _user_input.lower() == 'quit':
        print('[Chat] Exiting. An-Ra is still running — run Cell 5 again to reconnect.')
        break

    elif _user_input.lower() == 'url':
        if PUBLIC_URL:
            _current_url = f'{PUBLIC_URL}/chat'
            print(f'[Chat] Switched to public URL: {_current_url}\n')
        else:
            print('[Chat] No public URL available. Run Cell 5 first.\n')
        continue

    elif _user_input.lower() == 'local':
        _current_url = LOCAL_URL
        print(f'[Chat] Switched to local endpoint: {_current_url}\n')
        continue

    elif _user_input.lower() == 'clear':
        _history.clear()
        print('[Chat] Conversation history cleared.\n')
        continue

    elif _user_input.lower() == 'status':
        _pub = getattr(_b, 'ANRA_PUBLIC_URL', None)
        print(f'[Status] Endpoint      : {_current_url}')
        print(f'[Status] Public URL    : {_pub or "none"}')
        print(f'[Status] Checkpoint    : {getattr(_b, "ANRA_CHECKPOINT", "unknown")}')
        print(f'[Status] History turns : {len(_history)}')
        try:
            _ping = requests.get(f'http://localhost:{PORT}/health', timeout=3)
            print(f'[Status] Server health : HTTP {_ping.status_code} ✓')
        except Exception:
            print('[Status] Server health : ✗ Not responding')
        print()
        continue

    # ── Send message to An-Ra ─────────────────────────────────
    _t0 = time.time()
    _response = _send(_user_input, _current_url, _history)
    _latency = time.time() - _t0

    # Store in history for multi-turn context
    _history.append({'role': 'user',      'content': _user_input})
    _history.append({'role': 'assistant', 'content': _response})

    print(f'\nAn-Ra [{_latency:.1f}s]: {_response}\n')


---
## 💾 Cell 7 — SAVE & SHUTDOWN
*Copies all checkpoints and tokenizer back to Drive.*  
*Creates a timestamped backup for every file.*  
*Run this before closing the tab or when the session is about to timeout.*

In [ ]:
# ==============================================================
# CELL 7 — SAVE & SHUTDOWN
# Copies the latest artifacts from /content/anra to Drive.
# Creates versioned backups with UTC timestamps.
# Cleanly terminates the ngrok tunnel and server process.
# ==============================================================

import os, shutil, builtins as _b
from datetime import datetime, timezone

REPO_DIR  = getattr(_b, 'ANRA_REPO_DIR',  '/content/anra')
DRIVE_DIR = getattr(_b, 'ANRA_DRIVE_DIR', '/content/drive/MyDrive/AnRa')
_ts       = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')

# All artifacts to persist
ARTIFACTS = [
    ('anra_brain.pt',          'Model checkpoint (base)'),
    ('anra_brain_identity.pt', 'Model checkpoint (identity)'),
    ('tokenizer.pkl',          'Character tokenizer'),
]

print('='*52)
print(f'  💾  AN-RA SAVE — {_ts} UTC')
print('='*52)

_saved_any = False
_total_mb  = 0.0

for fname, desc in ARTIFACTS:
    local_path = os.path.join(REPO_DIR, fname)

    if not os.path.exists(local_path):
        print(f'  –  {fname:<30} (not found, skipping)')
        continue

    _size_mb = os.path.getsize(local_path) / 1e6
    _total_mb += _size_mb

    # ── Overwrite the canonical Drive copy ────────────────────
    drive_canonical = os.path.join(DRIVE_DIR, fname)
    shutil.copy2(local_path, drive_canonical)

    # ── Also write a timestamped backup ───────────────────────
    _stem, _ext = os.path.splitext(fname)
    drive_backup = os.path.join(DRIVE_DIR, f'{_stem}_{_ts}{_ext}')
    shutil.copy2(local_path, drive_backup)

    print(f'  ✓  {fname:<30} {_size_mb:>6.1f} MB  →  Drive')
    print(f'     Backup: {os.path.basename(drive_backup)}')
    _saved_any = True

if not _saved_any:
    print('  ✗  No checkpoint files found. Nothing saved.')
    print(f'     Expected artifacts in: {REPO_DIR}')

print('='*52)
print(f'  Total saved : {_total_mb:.1f} MB')
print(f'  Timestamp   : {_ts} UTC')
print('='*52)

# ── Cleanup: terminate server process ─────────────────────────
_proc = getattr(_b, 'ANRA_SERVER_PROC', None)
if _proc and _proc.poll() is None:   # process is still running
    _proc.terminate()
    print('\n[Shutdown] ✓ Server process terminated.')
else:
    print('\n[Shutdown]   Server process already stopped.')

# ── Cleanup: close ngrok tunnel ───────────────────────────────
try:
    from pyngrok import ngrok
    ngrok.kill()
    print('[Shutdown] ✓ ngrok tunnel closed.')
except Exception:
    pass    # pyngrok may not be imported if Cell 5 was never run

# ── List Drive backups ────────────────────────────────────────
try:
    _all_backups = sorted(
        [f for f in os.listdir(DRIVE_DIR) if f.endswith('.pt') and '_2' in f],
        reverse=True
    )
    if _all_backups:
        print(f'\n[Drive] Total backups on Drive: {len(_all_backups)}')
        print('[Drive] Latest 3:')
        for _bk in _all_backups[:3]:
            _bk_mb = os.path.getsize(os.path.join(DRIVE_DIR, _bk)) / 1e6
            print(f'        {_bk} ({_bk_mb:.1f} MB)')
except Exception:
    pass

print('\n[Shutdown] ✓ All done. Safe to close this tab.')
print('[Shutdown]   An-Ra is preserved on Google Drive.')
